### Rag Pipelines Data Ingestion to Vector DB Pipelines

In [8]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter 
from pathlib import Path
import os

In [3]:
# read all pdf files 
# add metadata 
# load them into document format 
from pathlib import Path
def process_all_pdfs(pdf_directory):

    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
 
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents laoded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")



Found 3 PDF files to process

Processing: 2021-02-17_cc_08-2021_transformation_stromerzeugung_2050_0.pdf
 Loaded 254 pages

Processing: DE2024_ISE_Studie_Stromgestehungskosten_Erneuerbare_Energien.pdf
 Loaded 52 pages

Processing: MonitoringberichtEnergie2024.pdf
 Loaded 348 pages

Total documents laoded: 654


In [4]:
#split documents into smaller chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")

    return split_docs


In [5]:
chunks = split_documents(all_pdf_documents)

Split 654 documents into 1874 chunks

Example chunk:
Content: CLIMATE CHANGE
08/202
 
1 
Für Mensch & Umwelt 
Strommarkt und 
Klimaschutz: 
Transformation der 
Stromerzeugung bis 
2050 
Abschlussbericht...
Metadata:{'producer': '2.2.13 (4.1.11)', 'creator': 'Acrobat PDFMaker 11 für Word', 'creationdate': '2020-12-18T12:49:39+01:00', 'author': 'Dr. Barbara Burstedde, Dr. Marco Nicolosi', 'category': '', 'comments': '', 'company': '', 'contenttypeid': '0x0101004A34EF8E09BCD6489E9276373EEBB436', 'keywords': 'Strommarkt, Energiesystemmodellierung, Klimaschutz, Zielszenarien, Sektorkopplung, Power to Heat, Power to Gas, Stromnetz, Optimierung, Electricity markets, energy modelling, climate protection, scenario, sector coupling, power to heat, power to gas, electricity network, optimization', 'moddate': '2021-02-24T13:37:11+01:00', 'sourcemodified': 'D:20201218114825', 'subject': 'Strommarkt und Klimaschutz: Transformation der Stromerzeugung bis 2050', 'title': 'Strommarkt und Klimaschutz: T

In [1]:
import numpy as np

from sentence_transformers import SentenceTransformer

import chromadb

from chromadb.config import Settings

import uuid

from typing import List, Dict, Any, Tuple

from sklearn.metrics.pairwise import cosine_similarity

c:\Github\rag_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            NumPy array of embeddings with shape
            (number_of_texts, embedding_dimension)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()

embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Github\rag_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\karst\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7753.43it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\karst\AppData\Local\Temp\ipykernel_23568\1243956011.py:26: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistent ChromaDB client
            os.makedirs(
                self.persist_directory,
                exist_ok=True
            )

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get existing collection or create a new one
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error initializing vector store: {e}"
            )
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        # Check that documents and embeddings have the same length
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match "
                "number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents "
            f"to vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        # Process every document together with its embedding
        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate a unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(
                doc.page_content
            )

            metadatas.append(metadata)

            # Store document text
            documents_text.append(
                doc.page_content
            )

            # Store embedding
            embeddings_list.append(
                embedding.tolist()
            )

        # Add everything to ChromaDB
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added "
                f"{len(documents)} documents "
                f"to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise


# Initialize the vector store
vectorstore = VectorStore()

vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
# convert text to embeddings
texts=[doc.page_content for doc in chunks]

embeddings=embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 1874 texts...


Batches: 100%|██████████| 59/59 [01:39<00:00,  1.68s/it]


Generated embeddings with shape: (1874, 384)
Adding 1874 documents to vector store...
Successfully added 1874 documents to vector store
Total documents in collection: 1874


In [17]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents
            and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(
            f"Top K: {top_k}, "
            f"Score threshold: {score_threshold}"
        )

        # Generate embedding for the query
        query_embedding = (
            self.embedding_manager
            .generate_embeddings([query])[0]
        )

        # Search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[
                    query_embedding.tolist()
                ],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # Convert distance to similarity score
                    similarity_score = 1 - distance

                    # Apply similarity threshold
                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents (after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Initialize the retriever
rag_retriever = RAGRetriever(
    vectorstore,
    embedding_manager
)

In [18]:
rag_retriever.retrieve("Erstelle einen eüberblick über erneuerbare Energie")

Retrieving documents for query: 'Erstelle einen eüberblick über erneuerbare Energie'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_33356c61_609',
  'content': 'insbesondere in Zeiten niedriger Preise bzw. eines großen (Überschuss-) Angebots von erneuer-\nbaren Energien eingesetzt werden, verzeichnen durch die höhere EE-Erzeugung einen Anstieg \nim Einsatz bzw. Stromverbrauch. Das gilt insbesondere im Jahr 2030, in dem der Effekt der \nnationalen EE-Ziele und die damit einhergehende regionale Konzentration von erneuerbaren',
  'metadata': {'page': 183,
   'keywords': 'Strommarkt, Energiesystemmodellierung, Klimaschutz, Zielszenarien, Sektorkopplung, Power to Heat, Power to Gas, Stromnetz, Optimierung, Electricity markets, energy modelling, climate protection, scenario, sector coupling, power to heat, power to gas, electricity network, optimization',
   'subject': 'Strommarkt und Klimaschutz: Transformation der Stromerzeugung bis 2050',
   'creator': 'Acrobat PDFMaker 11 für Word',
   'file_type': 'pdf',
   'sourcemodified': 'D:20201218114825',
   'page_label': '184',
   'company': '',
   'comments': ''